# Paper Summary Tables

Load all available LLM-judge analyses and prepare ensemble tables for the paper. Missing or unfinished Qwen artifacts are skipped independently, so completed metric summaries can be used even before Qwen pairwise win-rate evaluation finishes.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

REPO_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "evaluation_results.py").exists())
BASE_DIR = REPO_ROOT / "evaluations" / "biomni-base"
ANSWER_REPORT = "original"
WINRATE_DIRNAME = f"answer_winrate-{ANSWER_REPORT}"
JUDGE_IDS = ["old-llama-70b", "gpt-5.4-mini", "qwen3.6-35b-a3b"]
JUDGE_LABELS = {
    "old-llama-70b": "Llama 3.3 70B",
    "gpt-5.4-mini": "GPT-5.4 mini",
    "qwen3.6-35b-a3b": "Qwen 3.6 35B",
}

TABLE_OUTPUT_DIR = BASE_DIR / "paper_tables" / "judges" / "ensemble"
TABLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def read_csv_or_empty(path):
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()


def judge_analysis_dir(judge_id):
    return BASE_DIR / "analysis" / "judges" / judge_id

BASE_DIR

PosixPath('/home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base')

## Load and Aggregate Judge Results

Each `*_mean` value is the equal-weight mean of the available judge-level means. Each corresponding `*_std` is the sample standard deviation across those judge-level means, not the within-question standard deviation stored in an individual judge's source summary. Per-judge source rows are retained in `judge_sources` and exported below.

In [2]:
CATEGORY_NAMES = ["bool", "entity", "numeric"]
availability_rows = []
per_judge_summaries = {category: [] for category in CATEGORY_NAMES}
winrate_frames = []
pairwise_frames = []

for judge_id in JUDGE_IDS:
    analysis_dir = judge_analysis_dir(judge_id)
    for category in CATEGORY_NAMES:
        path = analysis_dir / category / "results_summary.csv"
        frame = read_csv_or_empty(path)
        availability_rows.append({
            "judge_id": judge_id,
            "judge": JUDGE_LABELS[judge_id],
            "artifact": f"{category}_summary",
            "available": not frame.empty,
            "path": str(path.relative_to(REPO_ROOT)),
        })
        if not frame.empty:
            frame.insert(0, "judge_id", judge_id)
            frame.insert(1, "judge", JUDGE_LABELS[judge_id])
            per_judge_summaries[category].append(frame)

    winrate_path = analysis_dir / WINRATE_DIRNAME / "answer_winrate_summary.csv"
    winrate = read_csv_or_empty(winrate_path)
    availability_rows.append({
        "judge_id": judge_id,
        "judge": JUDGE_LABELS[judge_id],
        "artifact": "answer_winrate_summary",
        "available": not winrate.empty,
        "path": str(winrate_path.relative_to(REPO_ROOT)),
    })
    if not winrate.empty:
        winrate.insert(0, "judge_id", judge_id)
        winrate.insert(1, "judge", JUDGE_LABELS[judge_id])
        winrate_frames.append(winrate)

    pairwise_path = analysis_dir / WINRATE_DIRNAME / "pairwise_answer_winrate.csv"
    pairwise = read_csv_or_empty(pairwise_path)
    availability_rows.append({
        "judge_id": judge_id,
        "judge": JUDGE_LABELS[judge_id],
        "artifact": "pairwise_answer_winrate",
        "available": not pairwise.empty,
        "path": str(pairwise_path.relative_to(REPO_ROOT)),
    })
    if not pairwise.empty:
        pairwise["judge_id"] = judge_id
        pairwise["judge"] = JUDGE_LABELS[judge_id]
        pairwise_frames.append(pairwise)

availability = pd.DataFrame(availability_rows)


def aggregate_judge_values(source, group_column, metric_columns):
    rows = []
    for group_name, group in source.groupby(group_column, dropna=False):
        counts = pd.to_numeric(group["evaluated_examples"], errors="coerce")
        row = {
            group_column: group_name,
            "judge_count": group["judge_id"].nunique(),
            "evaluated_examples": counts.min(),
            "evaluated_examples_min": counts.min(),
            "evaluated_examples_max": counts.max(),
        }
        for column in metric_columns:
            values = pd.to_numeric(group[column], errors="coerce")
            row[column] = values.mean()
            row[column.removesuffix("_mean") + "_std"] = values.std(ddof=1)
        rows.append(row)
    return pd.DataFrame(rows)


judge_sources = {}
summaries = {}
for category in CATEGORY_NAMES:
    if not per_judge_summaries[category]:
        continue
    source = pd.concat(per_judge_summaries[category], ignore_index=True)
    metric_columns = [column for column in source.columns if column.endswith("_mean")]
    judge_sources[category] = source
    summaries[category] = aggregate_judge_values(source, "run", metric_columns)

if winrate_frames:
    winrate_source = pd.concat(winrate_frames, ignore_index=True)
    winrate_rows = []
    for method, group in winrate_source.groupby("method", dropna=False):
        row = {
            "method": method,
            "judge_count": group["judge_id"].nunique(),
            "comparisons_min": group["comparisons"].min(),
            "comparisons_max": group["comparisons"].max(),
        }
        for column in ["wins", "losses", "ties", "winrate"]:
            values = pd.to_numeric(group[column], errors="coerce")
            row[f"{column}_mean"] = values.mean()
            row[f"{column}_std"] = values.std(ddof=1)
        winrate_rows.append(row)
    summaries["answer_winrate"] = pd.DataFrame(winrate_rows)
else:
    winrate_source = pd.DataFrame()

if pairwise_frames:
    summaries["pairwise_answer_winrate"] = pd.concat(pairwise_frames, ignore_index=True)

display(availability)
summaries.keys()

,judge_id,judge,artifact,available,path
0,old-llama-70b,Llama 3.3 70B,bool_summary,True,evaluations/biomni-base/analysis/judges/old-ll...
1,old-llama-70b,Llama 3.3 70B,entity_summary,True,evaluations/biomni-base/analysis/judges/old-ll...
2,old-llama-70b,Llama 3.3 70B,numeric_summary,True,evaluations/biomni-base/analysis/judges/old-ll...
3,old-llama-70b,Llama 3.3 70B,answer_winrate_summary,True,evaluations/biomni-base/analysis/judges/old-ll...
4,old-llama-70b,Llama 3.3 70B,pairwise_answer_winrate,True,evaluations/biomni-base/analysis/judges/old-ll...
5,gpt-5.4-mini,GPT-5.4 mini,bool_summary,True,evaluations/biomni-base/analysis/judges/gpt-5....
6,gpt-5.4-mini,GPT-5.4 mini,entity_summary,True,evaluations/biomni-base/analysis/judges/gpt-5....
7,gpt-5.4-mini,GPT-5.4 mini,numeric_summary,True,evaluations/biomni-base/analysis/judges/gpt-5....
8,gpt-5.4-mini,GPT-5.4 mini,answer_winrate_summary,True,evaluations/biomni-base/analysis/judges/gpt-5....
9,gpt-5.4-mini,GPT-5.4 mini,pairwise_answer_winrate,True,evaluations/biomni-base/analysis/judges/gpt-5....


dict_keys(['bool', 'entity', 'numeric', 'answer_winrate', 'pairwise_answer_winrate'])

In [3]:
# Quick preview of each loaded summary.
for name, df in summaries.items():
    print(f"\n{name}: {df.shape[0]} rows x {df.shape[1]} columns")
    display(df.head())


bool: 7 rows x 35 columns


,run,judge_count,evaluated_examples,evaluated_examples_min,evaluated_examples_max,answer_token_precision_mean,answer_token_precision_std,answer_token_recall_mean,answer_token_recall_std,answer_token_f1_mean,answer_token_f1_std,gt_entity_total_mean,gt_entity_total_std,gt_entity_covered_mean,gt_entity_covered_std,gt_entity_coverage_mean,gt_entity_coverage_std,bertscore_precision_mean,bertscore_precision_std,bertscore_recall_mean,bertscore_recall_std,bertscore_f1_mean,bertscore_f1_std,llm_completeness_mean,llm_completeness_std,llm_faithfulness_mean,llm_faithfulness_std,llm_relevance_mean,llm_relevance_std,llm_understanderbility_mean,llm_understanderbility_std,nli_entailment_max_mean,nli_entailment_max_std,bool_accuracy_mean,bool_accuracy_std
0,fullcontext,3,39,39,39,0.561577,0.000000e+00,0.501599,0.000000e+00,0.507475,0.0,1.769231,0.0,0.333333,0.0,0.117521,0.000000e+00,0.825013,0.0,0.856685,1.359740e-16,0.840080,0.000000e+00,0.098291,0.032669,0.109402,0.065137,0.652991,0.052694,0.920342,0.085050,0.597792,0.000000e+00,0.333333,0.000000e+00
1,grasp,3,39,39,39,0.217301,3.399350e-17,0.144743,0.000000e+00,0.159550,0.0,1.769231,0.0,0.153846,0.0,0.059829,8.498375e-18,0.832710,0.0,0.777822,1.359740e-16,0.802363,1.359740e-16,0.276991,0.019805,0.305803,0.025963,0.797413,0.078205,0.765969,0.076189,0.311397,6.798700e-17,0.384615,0.000000e+00
2,hipporag,3,39,39,39,0.082919,0.000000e+00,0.036299,0.000000e+00,0.044028,0.0,1.769231,0.0,0.410256,0.0,0.145299,0.000000e+00,0.727483,0.0,0.784602,0.000000e+00,0.754838,0.000000e+00,0.331017,0.039784,0.407377,0.030541,0.934458,0.030870,0.964732,0.040493,0.278777,6.798700e-17,0.487179,6.798700e-17
3,hypergraphrag,3,39,39,39,0.304457,0.000000e+00,0.687661,0.000000e+00,0.405765,0.0,1.769231,0.0,0.410256,0.0,0.164530,0.000000e+00,0.832518,0.0,0.861861,0.000000e+00,0.846417,0.000000e+00,0.577778,0.015509,0.568632,0.013989,0.892650,0.006649,0.884103,0.071127,0.477832,0.000000e+00,0.538462,0.000000e+00
4,llmbased,3,39,39,39,0.211613,6.509259e-17,0.326088,3.925231e-17,0.249819,0.0,1.769231,0.0,0.384615,0.0,0.138889,0.000000e+00,0.780733,0.0,0.802996,1.359740e-16,0.791610,0.000000e+00,0.061538,0.055173,0.034188,0.027417,0.191453,0.076251,0.523077,0.106433,0.164555,0.000000e+00,0.000000,0.000000e+00



entity: 7 rows x 51 columns


,run,judge_count,evaluated_examples,evaluated_examples_min,evaluated_examples_max,answer_token_precision_mean,answer_token_precision_std,answer_token_recall_mean,answer_token_recall_std,answer_token_f1_mean,answer_token_f1_std,gt_entity_total_mean,gt_entity_total_std,gt_entity_covered_mean,gt_entity_covered_std,gt_entity_coverage_mean,gt_entity_coverage_std,bertscore_precision_mean,bertscore_precision_std,bertscore_recall_mean,bertscore_recall_std,bertscore_f1_mean,bertscore_f1_std,llm_completeness_mean,llm_completeness_std,llm_faithfulness_mean,llm_faithfulness_std,llm_relevance_mean,llm_relevance_std,llm_understanderbility_mean,llm_understanderbility_std,nli_entailment_max_mean,nli_entailment_max_std,entity_gt_total_mean,entity_gt_total_std,entity_retrieved_final_total_mean,entity_retrieved_final_total_std,entity_retrieved_total_total_mean,entity_retrieved_total_total_std,entity_recall_final_mean,entity_recall_final_std,entity_precision_final_mean,entity_precision_final_std,entity_f1_final_mean,entity_f1_final_std,entity_recall_total_mean,entity_recall_total_std,entity_precision_total_mean,entity_precision_total_std,entity_f1_total_mean,entity_f1_total_std
0,fullcontext,3,46,46,46,0.375428,6.798700e-17,0.237902,4.807407e-17,0.233184,0.0,4.978261,0.0,0.347826,0.000000,0.111237,0.000000e+00,0.805008,0.000000e+00,0.812383,0.000000e+00,0.808287,0.0,0.047826,0.018091,0.061594,0.045927,0.316667,0.018743,0.918333,0.040778,0.441166,0.000000e+00,4.826087,0.0,4.804348,0.000000,5.847826,1.087792e-15,0.000000,0.000000,0.000000,0.000000e+00,NaN,NaN,0.000000,0.000000,0.000000,0.000000e+00,NaN,NaN
1,grasp,3,46,46,46,0.276368,0.000000e+00,0.070897,0.000000e+00,0.098111,0.0,4.978261,0.0,0.072464,0.012551,0.026515,6.560799e-03,0.831964,0.000000e+00,0.789693,1.359740e-16,0.809511,0.0,0.122319,0.086417,0.137190,0.051676,0.269388,0.110218,0.594356,0.230521,0.566474,0.000000e+00,4.826087,0.0,0.210145,0.363982,0.210145,3.639817e-01,0.007576,0.013122,0.093750,NaN,0.583333,NaN,0.007576,0.013122,0.093750,NaN,0.583333,NaN
2,hipporag,3,46,46,46,0.450032,6.798700e-17,0.164688,0.000000e+00,0.137284,0.0,4.978261,0.0,2.347826,0.000000,0.378093,6.798700e-17,0.751936,1.359740e-16,0.811002,0.000000e+00,0.779953,0.0,0.230254,0.056376,0.292971,0.036871,0.482029,0.093886,0.753768,0.126133,0.462035,0.000000e+00,4.826087,0.0,101.347826,0.000000,252.586957,0.000000e+00,0.273611,0.000000,0.012289,0.000000e+00,0.045944,5.637184e-17,0.730808,0.000000,0.010682,0.000000e+00,0.022297,5.610429e-17
3,hypergraphrag,3,46,46,46,0.299865,0.000000e+00,0.372397,6.798700e-17,0.246984,0.0,4.978261,0.0,1.086957,0.000000,0.270076,0.000000e+00,0.811944,0.000000e+00,0.820537,0.000000e+00,0.815499,0.0,0.413808,0.125847,0.439267,0.103600,0.681755,0.114932,0.817077,0.103336,0.448343,0.000000e+00,4.826087,0.0,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN
4,llmbased,3,46,46,46,0.421169,0.000000e+00,0.284986,0.000000e+00,0.288454,0.0,4.978261,0.0,0.782609,0.000000,0.160227,0.000000e+00,0.799983,0.000000e+00,0.811319,0.000000e+00,0.805241,0.0,0.160018,0.040664,0.128986,0.013059,0.354203,0.117690,0.544348,0.107728,0.426468,6.798700e-17,4.826087,0.0,13.347826,0.000000,14.565217,2.175584e-15,0.286616,0.000000,0.047826,8.498375e-18,0.132415,0.000000e+00,0.286616,0.000000,0.045995,8.498375e-18,0.128458,0.000000e+00



numeric: 7 rows x 39 columns


,run,judge_count,evaluated_examples,evaluated_examples_min,evaluated_examples_max,answer_token_precision_mean,answer_token_precision_std,answer_token_recall_mean,answer_token_recall_std,answer_token_f1_mean,answer_token_f1_std,gt_entity_total_mean,gt_entity_total_std,gt_entity_covered_mean,gt_entity_covered_std,gt_entity_coverage_mean,gt_entity_coverage_std,bertscore_precision_mean,bertscore_precision_std,bertscore_recall_mean,bertscore_recall_std,bertscore_f1_mean,bertscore_f1_std,llm_completeness_mean,llm_completeness_std,llm_faithfulness_mean,llm_faithfulness_std,llm_relevance_mean,llm_relevance_std,llm_understanderbility_mean,llm_understanderbility_std,nli_entailment_max_mean,nli_entailment_max_std,numeric_ground_truth_count_mean,numeric_ground_truth_count_std,numeric_predicted_count_mean,numeric_predicted_count_std,numeric_accuracy_mean,numeric_accuracy_std
0,fullcontext,3,21,21,21,0.485783,6.798700e-17,0.444218,0.0,0.458992,0.000000e+00,1.52381,0.0,0.000000,0.000000e+00,0.000000,0.000000e+00,0.830512,1.359740e-16,0.877141,0.000000e+00,0.852858,1.359740e-16,0.000000,0.000000,0.050794,0.087977,0.522222,0.240669,0.923175,0.096920,0.732726,0.0,1.52381,0.0,7.153846,0.0,0.000000,0.000000e+00
1,grasp,3,21,21,21,0.219303,0.000000e+00,0.126313,0.0,0.120454,0.000000e+00,1.52381,0.0,0.000000,0.000000e+00,0.000000,0.000000e+00,0.818294,0.000000e+00,0.791153,1.359740e-16,0.802189,0.000000e+00,0.438095,0.243603,0.488889,0.232766,0.496032,0.257235,0.620635,0.223826,0.645753,0.0,1.52381,0.0,254.500000,0.0,0.190476,3.399350e-17
2,hipporag,3,21,21,21,0.842986,0.000000e+00,0.060895,0.0,0.086942,5.637184e-17,1.52381,0.0,0.000000,0.000000e+00,0.000000,0.000000e+00,0.715124,0.000000e+00,0.767079,0.000000e+00,0.739899,0.000000e+00,0.225397,0.051361,0.195238,0.060796,0.725397,0.113622,0.852063,0.124832,0.404436,0.0,1.52381,0.0,142.052632,0.0,0.142857,0.000000e+00
3,hypergraphrag,3,21,21,21,0.464553,6.798700e-17,0.585086,0.0,0.500181,0.000000e+00,1.52381,0.0,0.190476,3.399350e-17,0.190476,3.399350e-17,0.849935,1.359740e-16,0.878390,0.000000e+00,0.863190,0.000000e+00,0.462751,0.046964,0.436455,0.066642,0.726190,0.066368,0.894286,0.114694,0.565253,0.0,1.52381,0.0,6.000000,0.0,0.095238,1.699675e-17
4,llmbased,3,21,21,21,0.333005,0.000000e+00,0.374226,0.0,0.317195,0.000000e+00,1.52381,0.0,0.000000,0.000000e+00,0.000000,0.000000e+00,0.789269,0.000000e+00,0.797059,0.000000e+00,0.793017,1.359740e-16,0.033333,0.021822,0.021429,0.017976,0.165079,0.133447,0.611905,0.136837,0.489231,0.0,1.52381,0.0,5.050000,0.0,0.142857,0.000000e+00



answer_winrate: 7 rows x 12 columns


,method,judge_count,comparisons_min,comparisons_max,wins_mean,wins_std,losses_mean,losses_std,ties_mean,ties_std,winrate_mean,winrate_std
0,LWE,2,636,636,526.5,41.719300,105.5,36.062446,4.0,5.656854,0.830975,0.061149
1,fullcontext,2,106,106,18.0,9.899495,88.0,9.899495,0.0,0.000000,0.169811,0.093391
2,grasp,2,106,106,11.0,11.313708,94.5,12.020815,0.5,0.707107,0.106132,0.110069
3,hipporag,2,106,106,14.0,5.656854,91.5,6.363961,0.5,0.707107,0.134434,0.056702
4,hypergraphrag,2,106,106,37.5,16.263456,65.5,12.020815,3.0,4.242641,0.367925,0.133416



pairwise_answer_winrate: 1272 rows x 19 columns


,ground_truth_id,ground_truth_qtype,question,ground_truth_answer,method_a,method_b,method_a_key,method_b_key,answer_a,answer_b,winner,judge_rationale,winning_method,method_a_score,method_b_score,judge_id,judge,evaluation_judge_id,evaluation_config_path
0,gt_0,"[""multi"", ""numeric""]","\nHow many ""experiment execution"" are there in...",The answer to the question is 12 unique execut...,fullcontext,LWE,fullcontext,ours,There are 16 distinct experiment executions re...,The provenance document lists two distinct exe...,method_b,Method B provides a more understandable explan...,LWE,0.0,1.0,old-llama-70b,Llama 3.3 70B,NaN,NaN
1,gt_1,"[""multi"", ""entity""]",\nIn what places do we utilize AI in this work...,\nThe ChatBS System utilizes AI for the follow...,fullcontext,LWE,fullcontext,ours,The workflow uses AI in four distinct stages w...,In this Biomni workflow the artificial‑intelli...,method_b,The answer provided by method_b is more faithf...,LWE,0.0,1.0,old-llama-70b,Llama 3.3 70B,NaN,NaN
2,gt_10,"[""multi"", ""entity""]",\nwhat are the output parameters of the functi...,\n The input parameters to Step 6 - Biomni ...,fullcontext,LWE,fullcontext,ours,The function used in Step 6 – Biomni LLM gener...,The program that implements **Step 6 – Biomni ...,method_b,The answer provided by method_b is more faithf...,LWE,0.0,1.0,old-llama-70b,Llama 3.3 70B,NaN,NaN
3,gt_100,"[""multi"", ""numeric""]","\nwhat is the number of output for the ""Step 6...","\n The number of output for the ""Step 6 - L...",fullcontext,LWE,fullcontext,ours,I cannot determine that number because the exe...,The execution identified by `http://testwebsit...,method_b,The answer from method_b is more understandabl...,LWE,0.0,1.0,old-llama-70b,Llama 3.3 70B,NaN,NaN
4,gt_101,"[""multi"", ""numeric""]","\nwhat is the number of output for the ""Step 6...","\n The number of output for the ""Step 6 - r...",fullcontext,LWE,fullcontext,ours,The graph does not contain the execution with ...,The “Step 6 – response output port” in the exe...,method_b,The answer from method_b is more understandabl...,LWE,0.0,1.0,old-llama-70b,Llama 3.3 70B,NaN,NaN


## Inspect Available Columns

Use this cell when choosing the placeholders below.

In [4]:
for name, df in summaries.items():
    print(f"\n{name}")
    for column in df.columns:
        print(f"  - {column}")


bool
  - run
  - judge_count
  - evaluated_examples
  - evaluated_examples_min
  - evaluated_examples_max
  - answer_token_precision_mean
  - answer_token_precision_std
  - answer_token_recall_mean
  - answer_token_recall_std
  - answer_token_f1_mean
  - answer_token_f1_std
  - gt_entity_total_mean
  - gt_entity_total_std
  - gt_entity_covered_mean
  - gt_entity_covered_std
  - gt_entity_coverage_mean
  - gt_entity_coverage_std
  - bertscore_precision_mean
  - bertscore_precision_std
  - bertscore_recall_mean
  - bertscore_recall_std
  - bertscore_f1_mean
  - bertscore_f1_std
  - llm_completeness_mean
  - llm_completeness_std
  - llm_faithfulness_mean
  - llm_faithfulness_std
  - llm_relevance_mean
  - llm_relevance_std
  - llm_understanderbility_mean
  - llm_understanderbility_std
  - nli_entailment_max_mean
  - nli_entailment_max_std
  - bool_accuracy_mean
  - bool_accuracy_std

entity
  - run
  - judge_count
  - evaluated_examples
  - evaluated_examples_min
  - evaluated_examples_m

## Editable Paper Table Columns

Edit these lists to decide what goes into each paper table. Keep `run` or `method` first if you want method names visible.

In [5]:
# These columns show ensemble means and between-judge sample standard deviations.
bool_columns = [
    "run",
    "judge_count",
    "evaluated_examples",
    "bertscore_f1_mean",
    "bertscore_f1_std",
    "nli_entailment_max_mean",
    "nli_entailment_max_std",
    "bool_accuracy_mean",
    "bool_accuracy_std",
]

entity_columns = [
    "run",
    "judge_count",
    "evaluated_examples",
    "answer_token_f1_mean",
    "answer_token_f1_std",
    "bertscore_f1_mean",
    "bertscore_f1_std",
    "nli_entailment_max_mean",
    "nli_entailment_max_std",
    "entity_recall_final_mean",
    "entity_recall_final_std",
    "entity_precision_final_mean",
    "entity_precision_final_std",
    "entity_f1_final_mean",
    "entity_f1_final_std",
    "entity_recall_total_mean",
    "entity_recall_total_std",
    "entity_precision_total_mean",
    "entity_precision_total_std",
    "entity_f1_total_mean",
    "entity_f1_total_std",
]

numeric_columns = [
    "run",
    "judge_count",
    "evaluated_examples",
    "answer_token_f1_mean",
    "answer_token_f1_std",
    "numeric_accuracy_mean",
    "numeric_accuracy_std",
    "bertscore_f1_mean",
    "bertscore_f1_std",
    "nli_entailment_max_mean",
    "nli_entailment_max_std",
]

answer_winrate_columns = [
    "method",
    "judge_count",
    "comparisons_min",
    "comparisons_max",
    "winrate_mean",
    "winrate_std",
]

In [6]:
# Optional display names. Add, remove, or rename as needed for the paper.
column_labels = {
    "run": "Method",
    "method": "Method",
    "judge_count": "Judges",
    "evaluated_examples": "N",
    "evaluated_examples_min": "N Min",
    "evaluated_examples_max": "N Max",
    "comparisons_min": "Comparisons Min",
    "comparisons_max": "Comparisons Max",
    "answer_token_f1_mean": "Answer Token F1",
    "answer_token_f1_std": "Answer Token F1 SD",
    "bertscore_f1_mean": "BERTScore F1",
    "bertscore_f1_std": "BERTScore F1 SD",
    "nli_entailment_max_mean": "NLI Entailment",
    "nli_entailment_max_std": "NLI Entailment SD",
    "bool_accuracy_mean": "Bool Accuracy",
    "bool_accuracy_std": "Bool Accuracy SD",
    "entity_recall_final_mean": "Entity Recall",
    "entity_recall_final_std": "Entity Recall SD",
    "entity_precision_final_mean": "Entity Precision",
    "entity_precision_final_std": "Entity Precision SD",
    "entity_f1_final_mean": "Entity F1",
    "entity_f1_final_std": "Entity F1 SD",
    "entity_recall_total_mean": "Entity Recall Total",
    "entity_recall_total_std": "Entity Recall Total SD",
    "entity_precision_total_mean": "Entity Precision Total",
    "entity_precision_total_std": "Entity Precision Total SD",
    "entity_f1_total_mean": "Entity F1 Total",
    "entity_f1_total_std": "Entity F1 Total SD",
    "numeric_accuracy_mean": "Numeric Accuracy",
    "numeric_accuracy_std": "Numeric Accuracy SD",
    "winrate_mean": "Win Rate",
    "winrate_std": "Win Rate SD",
}

method_order = ["GWB", "VSB", "GRASP", "HippoRAG", "HyperGRAG", "Ours"]

method_labels = {
    "fullcontext": "FCB",
    "grasp": "GRASP",
    "hipporag": "HippoRAG",
    "hypergraphrag": "HyperGRAG",
    "llmbased": "GWB",
    "LWE": "Ours",
    "ours": "Ours",
    "vectorsimilarity": "VSB",
}

def order_method_rows(table, method_column="Method"):
    ordered = table[table[method_column].isin(method_order)].copy()
    ordered[method_column] = pd.Categorical(
        ordered[method_column],
        categories=method_order,
        ordered=True,
    )
    sort_columns = [method_column]
    if "Question Type" in ordered.columns:
        sort_columns = ["Question Type", method_column]
    return ordered.sort_values(sort_columns).reset_index(drop=True)

count_columns = {"Judges", "N", "N Min", "N Max", "Comparisons Min", "Comparisons Max"}

def scale_result_columns(table):
    scaled = table.copy()
    for position, column in enumerate(scaled.columns):
        if column in count_columns:
            continue
        values = scaled.iloc[:, position]
        if pd.api.types.is_numeric_dtype(values):
            scaled.iloc[:, position] = values * 100
    return scaled

## Build Individual Tables

In [7]:
round_digits = 3

bool_table = summaries["bool"][bool_columns].copy()
bool_table["run"] = bool_table["run"].replace(method_labels)
bool_table = scale_result_columns(bool_table.rename(columns=column_labels)).round(round_digits)
bool_table = order_method_rows(bool_table)

entity_table = summaries["entity"][entity_columns].copy()
entity_table["run"] = entity_table["run"].replace(method_labels)
entity_table = scale_result_columns(entity_table.rename(columns=column_labels)).round(round_digits)
entity_table = order_method_rows(entity_table)

numeric_table = summaries["numeric"][numeric_columns].copy()
numeric_table["run"] = numeric_table["run"].replace(method_labels)
numeric_table = scale_result_columns(numeric_table.rename(columns=column_labels)).round(round_digits)
numeric_table = order_method_rows(numeric_table)



In [8]:
if "answer_winrate" in summaries:
    answer_winrate_table = summaries["answer_winrate"][answer_winrate_columns].copy()
    answer_winrate_table["method"] = answer_winrate_table["method"].replace(method_labels)
    answer_winrate_table = scale_result_columns(answer_winrate_table.rename(columns=column_labels)).round(round_digits)
    answer_winrate_table = order_method_rows(answer_winrate_table)
else:
    answer_winrate_table = pd.DataFrame(
        columns=[column_labels.get(column, column) for column in answer_winrate_columns]
    )

In [9]:
display(bool_table)
display(entity_table)
display(numeric_table)
display(answer_winrate_table)

,Method,Judges,N,BERTScore F1,BERTScore F1 SD,NLI Entailment,NLI Entailment SD,Bool Accuracy,Bool Accuracy SD
0,GWB,3,39,79.161,0.0,16.456,0.0,0.000,0.0
1,VSB,3,39,80.637,0.0,40.477,0.0,43.590,0.0
2,GRASP,3,39,80.236,0.0,31.140,0.0,38.462,0.0
3,HippoRAG,3,39,75.484,0.0,27.878,0.0,48.718,0.0
4,HyperGRAG,3,39,84.642,0.0,47.783,0.0,53.846,0.0
5,Ours,3,39,82.982,0.0,45.537,0.0,76.923,0.0


,Method,Judges,N,Answer Token F1,Answer Token F1 SD,BERTScore F1,BERTScore F1 SD,NLI Entailment,NLI Entailment SD,Entity Recall,Entity Recall SD,Entity Precision,Entity Precision SD,Entity F1,Entity F1 SD,Entity Recall Total,Entity Recall Total SD,Entity Precision Total,Entity Precision Total SD,Entity F1 Total,Entity F1 Total SD
0,GWB,3,46,28.845,0.0,80.524,0.0,42.647,0.0,28.662,0.000,4.783,0.0,13.242,0.000,28.662,0.000,4.600,0.0,12.846,0.0
1,VSB,3,46,22.512,0.0,81.182,0.0,41.907,0.0,6.692,0.000,3.244,0.0,18.780,0.000,15.581,0.000,3.774,0.0,13.190,0.0
2,GRASP,3,46,9.811,0.0,80.951,0.0,56.647,0.0,0.758,1.312,9.375,NaN,58.333,NaN,0.758,1.312,9.375,NaN,58.333,NaN
3,HippoRAG,3,46,13.728,0.0,77.995,0.0,46.204,0.0,27.361,0.000,1.229,0.0,4.594,0.000,73.081,0.000,1.068,0.0,2.230,0.0
4,HyperGRAG,3,46,24.698,0.0,81.550,0.0,44.834,0.0,0.000,0.000,NaN,NaN,NaN,NaN,0.000,0.000,NaN,NaN,NaN,NaN
5,Ours,3,46,27.612,0.0,81.307,0.0,40.878,0.0,17.898,0.164,0.372,0.0,1.828,0.001,37.386,0.000,2.131,0.0,6.851,0.0


,Method,Judges,N,Answer Token F1,Answer Token F1 SD,Numeric Accuracy,Numeric Accuracy SD,BERTScore F1,BERTScore F1 SD,NLI Entailment,NLI Entailment SD
0,GWB,3,21,31.720,0.0,14.286,0.0,79.302,0.0,48.923,0.0
1,VSB,3,21,18.169,0.0,0.000,0.0,78.821,0.0,49.384,0.0
2,GRASP,3,21,12.045,0.0,19.048,0.0,80.219,0.0,64.575,0.0
3,HippoRAG,3,21,8.694,0.0,14.286,0.0,73.990,0.0,40.444,0.0
4,HyperGRAG,3,21,50.018,0.0,9.524,0.0,86.319,0.0,56.525,0.0
5,Ours,3,21,48.821,0.0,52.381,0.0,86.289,0.0,55.705,0.0


,Method,Judges,Comparisons Min,Comparisons Max,Win Rate,Win Rate SD
0,GWB,2,106,106,8.491,9.339
1,VSB,2,106,106,15.094,14.676
2,GRASP,2,106,106,10.613,11.007
3,HippoRAG,2,106,106,13.443,5.670
4,HyperGRAG,2,106,106,36.792,13.342
5,Ours,2,636,636,83.097,6.115


## Combined Table Placeholder

This creates one compact table across question types. Edit `combined_columns` to choose the shared metrics.

In [10]:
combined_columns = [
    "task",
    "run",
    "judge_count",
    "evaluated_examples",
    "answer_token_f1_mean",
    "answer_token_f1_std",
    "bertscore_f1_mean",
    "bertscore_f1_std",
    "nli_entailment_max_mean",
    "nli_entailment_max_std",
]

combined_parts = []
for task_name in ["bool", "entity", "numeric"]:
    part = summaries[task_name].copy()
    part.insert(0, "task", task_name)
    combined_parts.append(part)

combined_table = pd.concat(combined_parts, ignore_index=True)
combined_table = combined_table[combined_columns].copy()
combined_table["run"] = combined_table["run"].replace(method_labels)
combined_table = scale_result_columns(combined_table.rename(columns={**column_labels, "task": "Question Type"})).round(round_digits)
combined_table = order_method_rows(combined_table)

display(combined_table)

,Question Type,Method,Judges,N,Answer Token F1,Answer Token F1 SD,BERTScore F1,BERTScore F1 SD,NLI Entailment,NLI Entailment SD
0,bool,GWB,3,39,24.982,0.0,79.161,0.0,16.456,0.0
1,bool,VSB,3,39,33.134,0.0,80.637,0.0,40.477,0.0
2,bool,GRASP,3,39,15.955,0.0,80.236,0.0,31.140,0.0
3,bool,HippoRAG,3,39,4.403,0.0,75.484,0.0,27.878,0.0
4,bool,HyperGRAG,3,39,40.576,0.0,84.642,0.0,47.783,0.0
5,bool,Ours,3,39,38.079,0.0,82.982,0.0,45.537,0.0
6,entity,GWB,3,46,28.845,0.0,80.524,0.0,42.647,0.0
7,entity,VSB,3,46,22.512,0.0,81.182,0.0,41.907,0.0
8,entity,GRASP,3,46,9.811,0.0,80.951,0.0,56.647,0.0
9,entity,HippoRAG,3,46,13.728,0.0,77.995,0.0,46.204,0.0


## Overall Summary Table

This combines bool, entity, and numeric summaries into one method-level table using only columns common to all three categories. Metric means are weighted by the number of evaluated examples in each category.


In [11]:
category_names = CATEGORY_NAMES
common_summary_columns = set.intersection(
    *(set(judge_sources[category].columns) for category in category_names)
)

overall_summary_columns = [
    "run",
    "evaluated_examples",
    "answer_token_f1_mean",
    "bertscore_f1_mean",
    "bertscore_f1_std",
    "nli_entailment_max_mean",
    "nli_entailment_max_std",
    "llm_completeness_mean",
    "llm_completeness_std",
    "llm_faithfulness_mean",
    "llm_faithfulness_std",
    "llm_relevance_mean",
    "llm_relevance_std",
    "llm_understanderbility_mean",
    "llm_understanderbility_std",
    "entity_recall_final_mean",
    "entity_precision_final_mean",
    "entity_f1_final_mean",
    "entity_recall_final_mean",
    "entity_recall_final_std",
    "entity_precision_final_mean",
    "entity_precision_final_std",
    "entity_f1_final_mean",
    "entity_f1_final_std",
    "entity_recall_total_mean",
    "entity_recall_total_std",
    "entity_precision_total_mean",
    "entity_precision_total_std",
    "entity_f1_total_mean",
    "entity_f1_total_std",
]
overall_summary_columns = [
    column for column in overall_summary_columns
    if column in common_summary_columns
    and (column in {"run", "evaluated_examples"} or column.endswith("_mean"))
]

overall_summary_source = pd.concat(
    [
        judge_sources[category][["judge_id", *overall_summary_columns]].assign(task=category)
        for category in category_names
    ],
    ignore_index=True,
)

overall_metric_columns = [
    column for column in overall_summary_columns
    if column not in {"run", "evaluated_examples"}
]

overall_rows = []
for (judge_id, run_name), run_group in overall_summary_source.groupby(["judge_id", "run"], dropna=False):
    weights = pd.to_numeric(
        run_group["evaluated_examples"],
        errors="coerce",
    ).fillna(0)
    row = {
        "judge_id": judge_id,
        "run": run_name,
        "evaluated_examples": weights.sum(),
    }

    for column in overall_metric_columns:
        values = pd.to_numeric(run_group[column], errors="coerce")
        valid = values.notna() & weights.gt(0)
        if valid.any():
            row[column] = (values[valid] * weights[valid]).sum() / weights[valid].sum()
        else:
            row[column] = values.mean()

    overall_rows.append(row)

overall_judge_source = pd.DataFrame(overall_rows)
overall_summary_table = aggregate_judge_values(
    overall_judge_source,
    "run",
    overall_metric_columns,
)
overall_summary_table["run"] = overall_summary_table["run"].replace(method_labels)
overall_summary_table = scale_result_columns(overall_summary_table.rename(columns=column_labels)).round(round_digits)
overall_summary_table = order_method_rows(overall_summary_table)

display(overall_summary_table)


,Method,Judges,N,N Min,N Max,Answer Token F1,Answer Token F1 SD,BERTScore F1,BERTScore F1 SD,NLI Entailment,NLI Entailment SD,llm_completeness_mean,llm_completeness_std,llm_faithfulness_mean,llm_faithfulness_std,llm_relevance_mean,llm_relevance_std,llm_understanderbility_mean,llm_understanderbility_std
0,GWB,3,106,106,106,27.993,0.0,79.780,0.0,34.254,0.0,9.869,2.817,7.280,0.820,25.686,7.551,54.991,11.067
1,VSB,3,106,106,106,25.560,0.0,80.514,0.0,42.862,0.0,22.052,1.049,26.539,4.425,58.022,4.628,90.947,7.367
2,GRASP,3,106,106,106,12.514,0.0,80.543,0.0,48.833,0.0,24.179,9.211,26.890,7.725,50.856,12.479,66.270,17.006
3,HippoRAG,3,106,106,106,9.300,0.0,76.278,0.0,38.320,0.0,26.636,0.882,31.570,0.862,69.670,1.741,85.086,8.588
4,HyperGRAG,3,106,106,106,35.557,0.0,83.632,0.0,48.235,0.0,48.383,6.365,48.631,4.880,76.815,3.841,85.703,9.291
5,Ours,3,106,106,106,35.665,0.0,82.911,0.0,45.530,0.0,67.461,6.268,65.237,8.420,90.920,6.185,82.348,8.386


## Export Tables

Exports CSV and LaTeX versions into `paper_tables/`. Comment out any table you do not need.

In [12]:
tables = {
    "bool_summary": bool_table,
    "entity_summary": entity_table,
    "numeric_summary": numeric_table,
    "answer_winrate_summary": answer_winrate_table,
    "combined_summary": combined_table,
    "overall_summary": overall_summary_table,
}

for table_name, table in tables.items():
    csv_path = TABLE_OUTPUT_DIR / f"{table_name}.csv"
    tex_path = TABLE_OUTPUT_DIR / f"{table_name}.tex"

    table.to_csv(csv_path, index=False)
    table.to_latex(tex_path, index=False, escape=False)

    print(f"Wrote {csv_path}")
    print(f"Wrote {tex_path}")

availability.to_csv(TABLE_OUTPUT_DIR / "judge_artifact_availability.csv", index=False)
for category, source in judge_sources.items():
    source.to_csv(TABLE_OUTPUT_DIR / f"{category}_summary_by_judge.csv", index=False)
if not winrate_source.empty:
    winrate_source.to_csv(TABLE_OUTPUT_DIR / "answer_winrate_summary_by_judge.csv", index=False)
if "pairwise_answer_winrate" in summaries:
    summaries["pairwise_answer_winrate"].to_csv(
        TABLE_OUTPUT_DIR / "pairwise_answer_winrate_by_judge.csv",
        index=False,
    )
print(f"Wrote per-judge sources and availability under {TABLE_OUTPUT_DIR}")


Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/paper_tables/judges/ensemble/bool_summary.csv
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/paper_tables/judges/ensemble/bool_summary.tex
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/paper_tables/judges/ensemble/entity_summary.csv
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/paper_tables/judges/ensemble/entity_summary.tex
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/paper_tables/judges/ensemble/numeric_summary.csv
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/paper_tables/judges/ensemble/numeric_summary.tex
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/paper_tables/judges/ensemble/answer_winrate_summary.csv
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/paper_tables/judges/ensemb

Wrote per-judge sources and availability under /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/paper_tables/judges/ensemble


## Optional: Pairwise Win Rate Details

In [13]:
if "pairwise_answer_winrate" in summaries:
    pairwise_table = summaries["pairwise_answer_winrate"].copy()
    print(f"Loaded {len(pairwise_table)} pairwise decisions across {pairwise_table['judge_id'].nunique()} judges")
    preview_columns = [
        column for column in [
            "judge",
            "ground_truth_id",
            "method_a_key",
            "method_b_key",
            "winning_method",
        ]
        if column in pairwise_table.columns
    ]
    display(pairwise_table[preview_columns].head(20))

Loaded 1272 pairwise decisions across 2 judges


,judge,ground_truth_id,method_a_key,method_b_key,winning_method
0,Llama 3.3 70B,gt_0,fullcontext,ours,LWE
1,Llama 3.3 70B,gt_1,fullcontext,ours,LWE
2,Llama 3.3 70B,gt_10,fullcontext,ours,LWE
3,Llama 3.3 70B,gt_100,fullcontext,ours,LWE
4,Llama 3.3 70B,gt_101,fullcontext,ours,LWE
5,Llama 3.3 70B,gt_102,fullcontext,ours,LWE
6,Llama 3.3 70B,gt_103,fullcontext,ours,LWE
7,Llama 3.3 70B,gt_104,fullcontext,ours,LWE
8,Llama 3.3 70B,gt_105,fullcontext,ours,LWE
9,Llama 3.3 70B,gt_11,fullcontext,ours,LWE


## Krippendorff Alpha Agreement

This final section measures how consistently the LLM judges assign the **completeness**, **relevance**, and **faithfulness** scores used in the paper tables. It computes **interval Krippendorff's alpha** directly from the item-level `[0, 1]` ratings in each judge's raw `results.csv`; squared-distance disagreement is appropriate because the attributes are numeric rather than unordered labels.

A rating item is aligned by question type, method, and ground-truth ID. The overall rows cover the same six methods retained in the paper tables, and the remaining rows show agreement for each method separately. Items with at least two available ratings are usable, so an absent or partially completed Qwen run does not prevent calculation. Alpha is `1` for perfect agreement, approximately `0` when agreement is no better than expected from the pooled ratings, and can be negative for systematic disagreement. This item-level reliability measure complements the between-judge standard deviations of the aggregate means above.

In [14]:
AGREEMENT_ATTRIBUTES = {
    "llm_completeness": "Completeness",
    "llm_relevance": "Relevance",
    "llm_faithfulness": "Faithfulness",
}
AGREEMENT_METHODS = [
    "llmbased",
    "vectorsimilarity",
    "grasp",
    "hipporag",
    "hypergraphrag",
    "ours",
]
ITEM_COLUMNS = ["question_type", "run", "ground_truth_id"]


def load_item_level_judge_ratings():
    frames = []
    rows = []
    for judge_id in JUDGE_IDS:
        for question_type in CATEGORY_NAMES:
            path = judge_analysis_dir(judge_id) / question_type / "results.csv"
            frame = read_csv_or_empty(path)
            available_attributes = [column for column in AGREEMENT_ATTRIBUTES if column in frame.columns]
            usable = not frame.empty and {"run", "ground_truth_id"}.issubset(frame.columns)
            rows.append({
                "judge_id": judge_id,
                "judge": JUDGE_LABELS[judge_id],
                "question_type": question_type,
                "available": usable,
                "rows": len(frame),
                "attributes": ", ".join(available_attributes),
                "path": str(path.relative_to(REPO_ROOT)),
            })
            if not usable or not available_attributes:
                continue
            selected = frame[["run", "ground_truth_id", *available_attributes]].copy()
            selected.insert(0, "question_type", question_type)
            selected.insert(0, "judge", JUDGE_LABELS[judge_id])
            selected.insert(0, "judge_id", judge_id)
            frames.append(selected)
    source = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    return source, pd.DataFrame(rows)


def krippendorff_alpha_interval(ratings):
    numeric = ratings.apply(pd.to_numeric, errors="coerce")
    usable = numeric.loc[numeric.notna().sum(axis=1).ge(2)]
    if usable.empty:
        return None
    observed_numerator = 0.0
    pooled_ratings = []
    for _, item_ratings in usable.iterrows():
        values = item_ratings.dropna().to_numpy(dtype=float)
        rating_count = len(values)
        centered_sum_squares = float(((values - values.mean()) ** 2).sum())
        observed_numerator += 2 * rating_count * centered_sum_squares / (rating_count - 1)
        pooled_ratings.extend(values)
    pooled = np.asarray(pooled_ratings, dtype=float)
    ratings_per_item = usable.notna().sum(axis=1)
    total_ratings = len(pooled)
    observed_disagreement = observed_numerator / total_ratings
    expected_disagreement = 2 * float(((pooled - pooled.mean()) ** 2).sum()) / (total_ratings - 1)
    alpha = 1 - observed_disagreement / expected_disagreement if expected_disagreement > 0 else np.nan
    return {
        "items_total": len(numeric),
        "items_used": len(usable),
        "ratings_used": total_ratings,
        "min_ratings_per_item": int(ratings_per_item.min()),
        "max_ratings_per_item": int(ratings_per_item.max()),
        "observed_disagreement": observed_disagreement,
        "expected_disagreement": expected_disagreement,
        "krippendorff_alpha": alpha,
    }


agreement_source, agreement_availability = load_item_level_judge_ratings()
alpha_rows = []
if not agreement_source.empty:
    agreement_source["ground_truth_id"] = agreement_source["ground_truth_id"].astype(str)
    agreement_source = agreement_source[agreement_source["run"].isin(AGREEMENT_METHODS)].copy()
    if agreement_source.duplicated(["judge_id", *ITEM_COLUMNS]).any():
        raise ValueError("Duplicate attribute ratings found for the same judge and item")

    for attribute, attribute_label in AGREEMENT_ATTRIBUTES.items():
        if attribute not in agreement_source.columns:
            continue
        rating_matrix = agreement_source.pivot(
            index=ITEM_COLUMNS,
            columns="judge_id",
            values=attribute,
        )
        scopes = [("overall", "all", "All paper methods", rating_matrix)]
        scopes.extend(
            (
                "method",
                method,
                method_labels.get(method, method),
                rating_matrix[rating_matrix.index.get_level_values("run") == method],
            )
            for method in AGREEMENT_METHODS
        )
        for scope_type, scope, scope_label, scoped_matrix in scopes:
            scope_judges = [
                judge_id for judge_id in JUDGE_IDS
                if judge_id in scoped_matrix.columns and scoped_matrix[judge_id].notna().any()
            ]
            stats = krippendorff_alpha_interval(scoped_matrix[scope_judges])
            if stats is not None:
                alpha_rows.append({
                    "attribute": attribute_label,
                    "scope_type": scope_type,
                    "scope": scope,
                    "scope_label": scope_label,
                    "distance": "interval_squared",
                    "judge_count": len(scope_judges),
                    "judges": ", ".join(scope_judges),
                    **stats,
                })

krippendorff_agreement = pd.DataFrame(alpha_rows)
alpha_path = TABLE_OUTPUT_DIR / "krippendorff_alpha_agreement.csv"
availability_path = TABLE_OUTPUT_DIR / "krippendorff_alpha_input_availability.csv"
krippendorff_agreement.to_csv(alpha_path, index=False)
agreement_availability.to_csv(availability_path, index=False)
print(f"Wrote {alpha_path}")
print(f"Wrote {availability_path}")
if krippendorff_agreement.empty:
    print("At least two judges with overlapping item-level attribute ratings are required for alpha.")
else:
    alpha_overall_table = krippendorff_agreement.loc[
        krippendorff_agreement["scope_type"].eq("overall"),
        ["attribute", "judge_count", "items_used", "ratings_used", "krippendorff_alpha"],
    ].rename(columns={
        "attribute": "Attribute",
        "judge_count": "Judges",
        "items_used": "Items",
        "ratings_used": "Ratings",
        "krippendorff_alpha": "Krippendorff Alpha",
    })
    overall_csv_path = TABLE_OUTPUT_DIR / "krippendorff_alpha_overall.csv"
    overall_tex_path = TABLE_OUTPUT_DIR / "krippendorff_alpha_overall.tex"
    alpha_overall_table.round(3).to_csv(overall_csv_path, index=False)
    alpha_overall_table.round(3).to_latex(overall_tex_path, index=False, escape=False)
    print(f"Wrote {overall_csv_path}")
    print(f"Wrote {overall_tex_path}")
    display(alpha_overall_table.round(3))
    display(krippendorff_agreement.loc[krippendorff_agreement["scope_type"].eq("method")].round(3))

Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/paper_tables/judges/ensemble/krippendorff_alpha_agreement.csv
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/paper_tables/judges/ensemble/krippendorff_alpha_input_availability.csv
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/paper_tables/judges/ensemble/krippendorff_alpha_overall.csv
Wrote /home/desild/work/research/LLM-Workflow-Explorer/evaluations/biomni-base/paper_tables/judges/ensemble/krippendorff_alpha_overall.tex


,Attribute,Judges,Items,Ratings,Krippendorff Alpha
0,Completeness,3,636,1903,0.811
7,Relevance,3,636,1903,0.680
14,Faithfulness,3,636,1903,0.760


,attribute,scope_type,scope,scope_label,distance,judge_count,judges,items_total,items_used,ratings_used,min_ratings_per_item,max_ratings_per_item,observed_disagreement,expected_disagreement,krippendorff_alpha
1,Completeness,method,llmbased,GWB,interval_squared,3,"old-llama-70b, gpt-5.4-mini, qwen3.6-35b-a3b",106,106,318,3,3,0.017,0.039,0.559
2,Completeness,method,vectorsimilarity,VSB,interval_squared,3,"old-llama-70b, gpt-5.4-mini, qwen3.6-35b-a3b",106,106,318,3,3,0.032,0.229,0.862
3,Completeness,method,grasp,GRASP,interval_squared,3,"old-llama-70b, gpt-5.4-mini, qwen3.6-35b-a3b",106,106,316,2,3,0.123,0.305,0.598
4,Completeness,method,hipporag,HippoRAG,interval_squared,3,"old-llama-70b, gpt-5.4-mini, qwen3.6-35b-a3b",106,106,317,2,3,0.053,0.269,0.803
5,Completeness,method,hypergraphrag,HyperGRAG,interval_squared,3,"old-llama-70b, gpt-5.4-mini, qwen3.6-35b-a3b",106,106,317,2,3,0.065,0.360,0.819
6,Completeness,method,ours,Ours,interval_squared,3,"old-llama-70b, gpt-5.4-mini, qwen3.6-35b-a3b",106,106,317,2,3,0.077,0.296,0.741
8,Relevance,method,llmbased,GWB,interval_squared,3,"old-llama-70b, gpt-5.4-mini, qwen3.6-35b-a3b",106,106,318,3,3,0.079,0.114,0.310
9,Relevance,method,vectorsimilarity,VSB,interval_squared,3,"old-llama-70b, gpt-5.4-mini, qwen3.6-35b-a3b",106,106,318,3,3,0.150,0.313,0.520
10,Relevance,method,grasp,GRASP,interval_squared,3,"old-llama-70b, gpt-5.4-mini, qwen3.6-35b-a3b",106,106,316,2,3,0.181,0.422,0.570
11,Relevance,method,hipporag,HippoRAG,interval_squared,3,"old-llama-70b, gpt-5.4-mini, qwen3.6-35b-a3b",106,106,317,2,3,0.106,0.307,0.655
